In [ ]:
from model_arch import *
from dataloading import *

In [ ]:
class Phase4ClassificationModel(nn.Module):
    def __init__(self, base_model, num_classes=2, pad_token_id=0):
        super().__init__()
        self.base_model = base_model
        self.num_classes = num_classes
        self.pad_token_id = pad_token_id
        self.classification_head = nn.Linear(self.base_model.cfg.d_model, num_classes)

    def forward(self, idx):
        B, T = idx.shape
        x = self.base_model.tok_emb(idx) + self.base_model.pos_emb[:T]
        for block in self.base_model.blocks:
            x = block(x)
        hidden_states = self.base_model.ln_f(x)
        
        sequence_lengths = (idx != self.pad_token_id).sum(dim=1)
        last_token_indices = torch.clamp(sequence_lengths - 1, min=0)
        tail_output = hidden_states[torch.arange(B, device=idx.device), last_token_indices]
        
        logits = self.classification_head(tail_output)
        return logits

class LoRMALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, r: int = 8, alpha: float = 8.0):
        super().__init__()
        self.base = base_layer
        self.scale = alpha / r
        
        for param in self.base.parameters():
            param.requires_grad = False
            
        d = base_layer.out_features 
        
        self.lora_A = nn.Linear(d, r, bias=False)
        self.lora_B = nn.Linear(r, d, bias=False)
        
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        z = self.base(x)
        return z + self.scale * self.lora_B(self.lora_A(z))

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
cfg = LMConfig(vocab_size=50260)
base_lm = ThirdDecoderLM(cfg)

model = Phase4ClassificationModel(base_lm, num_classes=2, pad_token_id=0)

rank = 8
alpha_val = 8.0

for block in model.base_model.blocks:
    block.attn.q_proj = LoRMALinear(block.attn.q_proj, r=rank, alpha=alpha_val)
    block.attn.v_proj = LoRMALinear(block.attn.v_proj, r=rank, alpha=alpha_val)

model.load_state_dict(torch.load("checkpoint.pt", map_location=device))
model.to(device)
model.eval()

In [ ]:
_, _, test_data = load_cjpe()

In [ ]:
predictions = []

with torch.no_grad():
    for doc in test_data:
        doc_id = doc.get('id')
        encoded_doc = doc['tokens'][-1024:]
        input_tensor = torch.tensor(encoded_doc, dtype=torch.long).unsqueeze(0).to(device)
        logits = model(input_tensor)
        pred_val = torch.argmax(logits, dim=1).item()
        predictions.append({"id": doc_id, "pred": pred_val})

with open("prediction.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=4)